##### Copyright 2026 Google LLC.

In [4]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multimodal Live API - Translation Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_LiveTranslate.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

**Preview**: The Live API is in preview.

This notebook demonstrates usage of the Gemini Live API for real-time audio translation. For an overview of new capabilities refer to the [Gemini Live API docs](https://ai.google.dev/gemini-api/docs/live-api/capabilities).

Some features of the API (such as low-latency bidirectional voice and video streaming using the local microphone and camera) are not supported in a standard Colab environment due to its headless cloud VM nature. To try full local hardware streaming, check out the CLI examples in the [Cookbook repository](https://github.com/google-gemini/cookbook/tree/main/quickstarts).

In this notebook, you will learn how to **stream and translate audio from a URL** in real-time using the Live Translation API, displaying live transcripts and playing the translated audio output.

## Setup

### Install SDK and Dependencies

The new **[Google Gen AI SDK](https://ai.google.dev/gemini-api/docs/sdks)** provides programmatic access to Gemini.

> **Note**: This notebook also uses `ffmpeg` to process the audio stream. `ffmpeg` is pre-installed in Google Colab environments.

In [5]:
%pip install -U -q google-genai

### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for details.

In [6]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('first_key')

### Initialize SDK client

The client will pick up your API key from the environment variable.

In [7]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

### Select a model

The Live Translation API uses the translation-capable Live models.

In [8]:
MODEL = 'gemini-3.5-live-translate-preview'  # @param ['gemini-3.5-live-translate-preview'] {allow-input: true, isTemplate: true}

### Import Modules

Import the necessary packages for handling async events, audio format streams, and file writing.

In [9]:
import array
import asyncio
import contextlib
import wave

from IPython.display import display, Audio

from google import genai
from google.genai import types

### Helper to Write WAV Files

Let's define a helper context manager to write received audio chunks to a `.wav` file for playback in the notebook:

In [10]:
@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

## Audio URL Streaming & Translation

You can stream audio in real-time to the Live API, and receive translated audio back. Here, we'll stream audio from a public audio URL in 100ms chunks to mimic real-time audio input, and stream translation responses back.

### Helper for Streaming Audio URL

We define a helper function to stream audio from an HTTP URL and use `ffmpeg` to transcode it to raw PCM 16kHz mono audio.

In [11]:
async def stream_audio_url(url: str, audio_queue: asyncio.Queue, sample_rate: int = 16000, channels: int = 1, chunk_size: int = 1600):
    """Streams audio from an HTTP URL, decoding it via ffmpeg and putting raw PCM bytes into the audio_queue."""
    print(f"\n[Info] Starting audio stream via ffmpeg from: {url}")
    # Spawn ffmpeg to decode stream to raw PCM 16kHz mono 16-bit
    process = await asyncio.create_subprocess_exec(
        'ffmpeg',
        '-i', url,
        '-f', 's16le',
        '-acodec', 'pcm_s16le',
        '-ar', str(sample_rate),
        '-ac', str(channels),
        '-',
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.DEVNULL
    )

    # 1600 samples * 2 bytes/sample (16-bit) = 3200 bytes per chunk
    chunk_size_bytes = chunk_size * 2
    bytes_per_second = sample_rate * 2
    start_time = asyncio.get_event_loop().time()
    bytes_sent = 0

    try:
        while True:
            data = await process.stdout.read(chunk_size_bytes)
            if not data:
                break

            await audio_queue.put(data)
            bytes_sent += len(data)

            # Rate limit to real-time speed (1.0x) so we simulate real mic streaming
            expected_elapsed = bytes_sent / bytes_per_second
            actual_elapsed = asyncio.get_event_loop().time() - start_time
            sleep_time = expected_elapsed - actual_elapsed
            if sleep_time > 0:
                await asyncio.sleep(sleep_time)
    except asyncio.CancelledError:
        pass
    finally:
        if process.returncode is None:
            try:
                process.terminate()
                await process.wait()
            except Exception:
                pass
        print("\n[Info] Audio stream finished.")

### Helper for Sending Audio & Receiving Translated Responses

Next, we define functions to push the audio chunks from the queue to the Live session, and to receive and print the source and translation transcripts.

In [12]:
async def send_realtime(session, audio_queue: asyncio.Queue, sample_rate: int = 16000):
    """Sends audio from the input queue to the GenAI session."""
    try:
        while True:
            chunk = await audio_queue.get()
            await session.send_realtime_input(
                audio=types.Blob(
                    data=chunk,
                    mime_type=f"audio/pcm;rate={sample_rate}"
                )
            )
            audio_queue.task_done()
    except asyncio.CancelledError:
        pass

### Run Audio URL Translation

Now, we set up our main translation runner. We'll use a static audio URL: `https://storage.googleapis.com/generativeai-downloads/gemini-cookbook/audio/gemini-live-translate-sample.wav`.
The code runs the input audio stream, upload stream, and receiver concurrently in a `TaskGroup`. The translated Spanish audio is written to a wave file and played back.

In [15]:
import base64
import asyncio
import IPython
from google.colab import output
from google.genai import types
from IPython.display import display, Audio

# Configuration using BCP-47 language codes
SOURCE_LANGUAGE = "es-ES"
TARGET_LANGUAGE = "la"

async def run_mic_translation():
    # Initialize the GenAI Live Session
    async with client.aio.live.connect(
        model=MODEL,
        config=types.LiveConnectConfig(
            translation_config=types.TranslationConfig(
                source_language_code=SOURCE_LANGUAGE,
                target_language_code=TARGET_LANGUAGE
            )
        )
    ) as session:

        print(f"--- Listening... Speak in {SOURCE_LANGUAGE} ---")
        print(f"--- Translating to {TARGET_LANGUAGE} voice ---")

        # Audio queue to handle incoming chunks from JS
        audio_in_queue = asyncio.Queue()

        # Python function callable from JS to receive audio chunks
        def _receive_audio(audio_base64):
            audio_bytes = base64.b64decode(audio_base64)
            audio_in_queue.put_nowait(audio_bytes)

        output.register_callback('notebook.receive_audio', _receive_audio)

        # JavaScript to capture mic and send to Python
        js_code = """
        (async () => {
            const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
            const audioContext = new AudioContext({ sampleRate: 16000 });
            const source = audioContext.createMediaStreamSource(stream);
            const processor = audioContext.createScriptProcessor(4096, 1, 1);

            processor.onaudioprocess = (e) => {
                const inputData = e.inputBuffer.getChannelData(0);
                // Convert Float32 to Int16
                const pcmData = new Int16Array(inputData.length);
                for (let i = 0; i < inputData.length; i++) {
                    pcmData[i] = Math.max(-1, Math.min(1, inputData[i])) * 0x7FFF;
                }
                const base64 = btoa(String.fromCharCode(...new Uint8Array(pcmData.buffer)));
                google.colab.kernel.invokeFunction('notebook.receive_audio', [base64], {});
            };

            source.connect(processor);
            processor.connect(audioContext.destination);
            window.stopMic = () => {
                stream.getTracks().forEach(t => t.stop());
                processor.disconnect();
            };
        })();
        """
        display(IPython.display.Javascript(js_code))

        async def send_loop():
            try:
                while True:
                    chunk = await audio_in_queue.get()
                    await session.send_realtime_input(
                        audio=types.Blob(data=chunk, mime_type="audio/pcm;rate=16000")
                    )
            except asyncio.CancelledError:
                pass

        async def receive_loop():
            async for message in session.receive():
                if message.server_content and message.server_content.model_turn:
                    parts = message.server_content.model_turn.parts
                    for part in parts:
                        if part.inline_data:
                            # Play the translated audio returned by the API
                            display(Audio(part.inline_data.data, rate=24000, autoplay=True))
                        if part.text:
                            print(f"Translation: {part.text}")

        try:
            await asyncio.gather(send_loop(), receive_loop())
        except Exception as e:
            print(f"Session ended: {e}")
        finally:
            display(IPython.display.Javascript("window.stopMic();"))

# Run the async loop
await run_mic_translation()

ValidationError: 2 validation errors for TranslationConfig
source_language
  Extra inputs are not permitted [type=extra_forbidden, input_value='es', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
target_language
  Extra inputs are not permitted [type=extra_forbidden, input_value='la', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden

## Next steps

This tutorial shows basic audio translation capabilities using the Multimodal Live API.

- Try it out in [Google AI Studio](https://aistudio.google.com/live?model=gemini-3.5-live-translate-preview)
- Read the [docs](https://ai.google.dev/gemini-api/docs/live-api/live-translate)
- Clone the [Live API examples from GitHub](https://github.com/google-gemini/gemini-live-api-examples)
